In [ ]:
# TODO: Finish cleaning up national dex transformations 
# TODO: Would it be better to move the regional pokedex transformations under the scrape function?
# TODO: Consider moving constants and imports in all notebooks to a single separate notebook
# TODO: Modify conditional filtering by gen cap; relook at alternate forms in national dex
# TODO: Consider separating regional pokedexes by game (e.g. Johto: Gold/Silver v HeartGold/Soulsilver) - will require changes to REGIONAL_POKEDEX_URLS, REGIONAL_INDEX_CAP, and national dex
# TODO: UI enhancement: display dictionary at gen cap widget showing games and the gens they correspond to

### *This notebook serves as the master Pokemon data notebook. Other notebooks that are more narrowly focused on specific Pokemon data will run this notebook and use its functionality to simplify code writing and streamline functions like joining and webscraping.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
# Maximize display of all dataframes
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('future.no_silent_downcasting', True)

### Constants

In [ ]:
GEN_RANGES = {
    1: range(1, 152),
    2: range(152, 252),
    3: range(251, 387),
    4: range(386, 494),
    5: range(493, 650),
    6: range(649, 722),
    7: range(721, 810),
    8: range(809, 906),
    9: range(905, 1026)
}

# Add new gens as necessary
RENAME_ALTERNATE_FORMS = {
    'GIRATINA ALTERED FORME': 'GIRATINA',
    'GIRATINA ORIGIN FORME': 'GIRATINA DISTORTION FORME',
    'DARMANITAN STANDARD MODE': 'DARMANITAN',
    'BASCULIN RED-STRIPED FORM': 'BASCULIN',
    'TORNADUS INCARNATE FORME': 'TORNADUS',
    'THUNDURUS INCARNATE FORME': 'THUNDURUS',
    'LANDORUS INCARNATE FORME': 'LANDORUS',
    'KYUREM WHITE KYUREM': 'WHITE KYUREM',
    'KYUREM BLACK KYUREM': 'BLACK KYUREM',
    'MELOETTA ARIA FORME': 'MELOETTA',
    'KELDEO ORDINARY FORM': 'KELDEO'
}

ALTERNATE_FORMS = {
    'MEGA': 6, 
    'GALARIAN': 8, 
    'HISUIAN': 8, 
    'ALOLAN': 7, 
    'PALDEAN': 9, 
    'PRIMAL': 6, 
    'PARTNER': 'remove', 
    'BREED': 9, 
    'ORIGIN': 8
    }

REDUNDANT_POKEMON = ['BASCULIN BLUE-STRIPED FORM', 'BASCULIN WHITE-STRIPED FORM', 'KELDEO RESOLUTE FORM']

# Add new gens 
REGIONAL_POKEDEX_URLS = {
    'Johto': 'https://bulbapedia.bulbagarden.net/wiki/List_of_Pok%C3%A9mon_by_Johto_Pok%C3%A9dex_number',  # Based on gen 4 remakes
    'Hoenn': 'https://bulbapedia.bulbagarden.net/wiki/List_of_Pok%C3%A9mon_by_Hoenn_Pok%C3%A9dex_number_in_Generation_III',
    'Sinnoh': 'https://bulbapedia.bulbagarden.net/wiki/List_of_Pok%C3%A9mon_by_Sinnoh_Pok%C3%A9dex_number',
    'Unova (Black/White)':' https://bulbapedia.bulbagarden.net/wiki/List_of_Pok%C3%A9mon_by_Unova_Pok%C3%A9dex_number_in_Pok%C3%A9mon_Black_and_White',
    'Unova (Black2/White2)': 'https://bulbapedia.bulbagarden.net/wiki/List_of_Pok%C3%A9mon_by_Unova_Pok%C3%A9dex_number_in_Pok%C3%A9mon_Black_2_and_White_2',
}

# Add new gens
# Used to filter out unnecessary row data from each regional pokedex
REGIONAL_INDEX_CAP = {
    'Johto': 288,  # Based on gen 4 remake
    'Hoenn': 210,
    'Sinnoh': 253,
    'Unova (Black/White)': 172,
    'Unova (Black2/White2)': 329
}

# Correct namings in regional pokedex dataframes prior to national dex join
REGIONAL_POKEDEX_RENAMES = {
    'UNOWNONE FORM': 'UNOWN',
    'BURMYPLANT CLOAK': 'BURMY PLANT CLOAK',
    'WORMADAMPLANT CLOAK': 'WORMADAM PLANT CLOAK',
    'SHELLOSWEST SEA': 'SHELLOS',
    'CHERRIMOVERCAST FORM': 'CHERRIM',
    'GASTRODONWEST SEA': 'GASTRODON',
    'ROTOMROTOM': 'ROTOM',
    'GIRATINAALTERED FORME': 'GIRATINA',
    'UNFEZANTMALE': 'UNFEZANT',
    'BASCULINRED-STRIPED FORM': 'BASCULIN',
    'DARMANITANSTANDARD MODE': 'DARMANITAN',
    'DEERLINGSPRING FORM': 'DEERLING',
    'SAWSBUCKSPRING FORM': 'SAWSBUCK',
    'MELOETTAARIA FORME': 'MELOETTA',
    'CASTFORMNORMAL': 'CASTFORM',
    'FRILLISHMALE': 'FRILLISH',
    'JELLICENTMALE': 'JELLICENT',
    'TORNADUSINCARNATE FORME': 'TORNADUS',
    'THUNDURUSINCARNATE FORME': 'THUNDURUS',
    'LANDORUSINCARNATE FORME': 'LANDORUS',
    'KYUREMKYUREM': 'KYUREM',
    'KELDEOORDINARY FORM': 'KELDEO'
    
}

# To add to the appropriate regional pokedexes
MISSING_POKEMON = {
    'Sinnoh': ['BURMY SANDY CLOAK', 'BURMY TRASH CLOAK', 'WORMADAM SANDY CLOAK', 'WORMADAM TRASH CLOAK', 'HEAT ROTOM', 'WASH ROTOM', 'MOW ROTOM', 'FROST ROTOM', 'FAN ROTOM',
               'GIRATINA DISTORTION FORME'],
    'Unova (Black/White)': ['DARMANITAN ZEN MODE', 'MELOETTA PIROUETTE FORME', 'TORNADUS THERIAN FORME', 'THUNDURUS THERIAN FORME', 'LANDORUS THERIAN FORME',
                            'WHITE KYUREM', 'BLACK KYUREM'],
    'Unova (Black2/White2)': ['CASTFORM SUNNY FORM', 'CASTFORM RAINY FORM', 'CASTFORM SNOWY FORM', 'DARMANITAN ZEN MODE', 'TORNADUS THERIAN FORME', 'THUNDURUS THERIAN FORME', 
                              'LANDORUS THERIAN FORME', 'WHITE KYUREM', 'BLACK KYUREM']
}

### Web Scraping Function

In [ ]:
# Scrape data from a web server
def scrape_table(url):
    
    # Exception handling for connection errors
    try:
        response = requests.get(url)
        html = response.text

    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error occurred: {e}")

    except requests.exceptions.ConnectionError as e:
        print(f"Connection Error occurred: {e}.")

    except Exception as e:
        print(f"An unexpected error occurred: {e}")

    soup = BeautifulSoup(html, 'html.parser')
    rows = soup.find_all('tr') # Retrieve all rows within the database
    
    data = []

    for row in rows:
        cols = row.find_all(['td','th']) # Retrieve all columns within each row
        row_data = []
        
        # Account for columns that may contain an images as values
        for col in cols:
            img = col.find('img')  # Check for an image inside each cell
            if img and not col.text.strip():
                img_url = f'https://www.serebii.net{img['src']}' # Get full image url 
                row_data.append(img_url) 
            else:
                row_data.append(col.text.strip())  # Otherwise, get text
        data.append(row_data)
    
    global df
    df = pd.DataFrame(data)

### Join Function

In [ ]:
# Add new columns to an existing dataframe 
def join(df_main, df_join, kind, key):
    valid_kinds = ['left', 'right', 'inner', 'outer']
    if kind not in valid_kinds:
        raise ValueError(f'Kind must be of {valid_kinds}')
    df_main = pd.merge(df_main, df_join, how = kind, on = key)
    return df_main

### Pass National Pokedex URL into Scraping Function and Prepare Dataframe for Cleaning

In [ ]:
pokedex_url = 'https://pokemondb.net/pokedex/all'
scrape_table(pokedex_url)
pokedex_df = df

# Realign dataframe
pokedex_df.columns = pokedex_df.iloc[0] 
pokedex_df = pokedex_df.iloc[1:]

In [ ]:
# Placeholder while working directly out of this notebook - defined in Widgets
generation_cap = 5

### Apply Transformations to National Pokedex

In [ ]:
pokedex_df = pokedex_df.rename(columns = {'Total':'Base Stats', 'Name':'Pokémon'})
pokedex_df['Type'] = pokedex_df['Type'].str.replace(' ','|')
pokedex_df['#'] = pokedex_df['#'].astype('int')

# Assign gen based on pokedex 
def assign_gen(input):
    for gen, range in GEN_RANGES.items():
        if input in range:
            return gen
pokedex_df['Generation'] = pokedex_df['#'].apply(assign_gen).astype('int')

# Rename alternate form names for easier pokemon stats widget querying
pokedex_df['Pokémon'] = pokedex_df['Pokémon'].str.upper().str.strip().replace(RENAME_ALTERNATE_FORMS)

# Remove redundant pokemon namings
pokedex_df = pokedex_df[~pokedex_df['Pokémon'].isin(REDUNDANT_POKEMON)]

# Filter alternate forms based on gen cap
for form, gen in ALTERNATE_FORMS.items():
    if gen == 'remove' or gen > generation_cap:
        pokedex_df = pokedex_df[~pokedex_df['Pokémon'].str.contains(form)]

# Modify pre gen 3 typings - research this
if generation_cap < 3:
    pokedex_df['Type'] = pokedex_df['Type'].str.replace(r'(^Type\||\|?Type)', '', regex=True).replace('','Type')

# Fairy type exists only after gen 5
if generation_cap < 6:
    pokedex_df['Type'] = pokedex_df['Type'].str.replace(r'(^Fairy\||\|?Fairy)', '', regex=True).replace('','Normal')
    pokedex_df.loc[pokedex_df['Pokémon'].isin(['TOGETIC', 'TOGEKISS']), 'Type'] = 'Normal|Flying'

pokedex_df = pokedex_df[pokedex_df['Generation'] <= generation_cap]

# Assign Boolean to Pokemon that exist in the regional Kanto pokedex from the video game
pokedex_df['Kanto Pokedex'] = np.where(pokedex_df['#'] <= 151, True, False)

### Apply Regional Pokedex Booleans to National Pokedex

In [ ]:
for region, link in REGIONAL_POKEDEX_URLS.items():
    scrape_table(link)
    
    # Apply transformations
    for idx, row in enumerate(df.values.tolist()):
        if 'Pokémon' in row:
            df.columns = df.iloc[idx]
            df = df.iloc[(idx + 1):]
            break

    df = df[['Pokémon']].dropna()

    # Clean up malformatted pokemon names
    for old, new in REGIONAL_POKEDEX_RENAMES.items():
        df['Pokémon'] = df['Pokémon'].str.upper().replace({old: new}).str.strip()

    df = df[df['Pokémon'] != 'POKÉMON']
    df = df[df.index <= REGIONAL_INDEX_CAP[region]]

    # Add missing pokemon to regional pokedex
    for sub_region, missing_list in MISSING_POKEMON.items():
        if sub_region == region:
            missing_df = pd.DataFrame({'Pokémon': missing_list})
            df = pd.concat([df, missing_df])

    df[f'{region} Pokedex'] = True

    # Join booleans to national dex
    pokedex_df = join(pokedex_df, df, 'left', 'Pokémon')
    
for col in pokedex_df.columns:
    pokedex_df[col] = pokedex_df[col].fillna(False).infer_objects(copy = False)